# 🔬 Lab: ChromaDB + RAG — Busqueda semantica y embeddings visuales

**Workshop: Mas alla de SQL** | ITESM — Inteligencia de Negocios

En este notebook vas a:
1. Crear una base de datos vectorial con ChromaDB
2. Hacer busquedas semanticas (por significado, no por palabras exactas)
3. Visualizar los embeddings en 2D con PCA y t-SNE
4. Ver como funciona RAG (Retrieval-Augmented Generation)

---

**Setup**: Este notebook esta diseñado para Google Colab. No necesitas instalar nada en tu computadora.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HesusG/mas-alla-de-sql/blob/main/labs/lab-chroma-rag.ipynb)

In [ ]:
# Instalar dependencias (solo necesario en Colab)
!pip install -q chromadb scikit-learn matplotlib numpy

## Seccion 1 — ChromaDB: tu primera base de datos vectorial

ChromaDB convierte texto en **vectores** (numeros que representan significado) y permite buscar por **similitud semantica**.

Vamos a crear una coleccion con 15 documentos relacionados con Inteligencia de Negocios.

In [ ]:
import chromadb

# Crear cliente en memoria (no necesita servidor)
client = chromadb.Client()

# Crear coleccion — ChromaDB genera los embeddings automaticamente
collection = client.create_collection("bi_knowledge")

# 15 documentos sobre temas de BI — en español
documents = [
    # Programacion
    "Python es el lenguaje mas usado en ciencia de datos y machine learning",
    "SQL permite consultar bases de datos relacionales con SELECT, FROM y WHERE",
    "R es un lenguaje estadistico popular en investigacion academica",
    # Datos
    "Un Data Warehouse centraliza datos de multiples fuentes para analisis",
    "ETL significa Extract, Transform, Load — el proceso de preparar datos",
    "Los datos no estructurados como emails y chats representan el 80% de los datos empresariales",
    # Visualizacion
    "Tableau y Power BI son las herramientas lideres para crear dashboards interactivos",
    "Un buen dashboard cuenta una historia con los datos sin necesidad de explicacion",
    "La visualizacion de datos transforma numeros en decisiones de negocio",
    # Negocio
    "KPIs son metricas clave que miden el desempeño de un area del negocio",
    "Business Intelligence ayuda a tomar decisiones basadas en datos, no en intuicion",
    "El analisis predictivo usa datos historicos para anticipar tendencias futuras",
    # Machine Learning
    "Los modelos de machine learning aprenden patrones automaticamente de los datos",
    "Los embeddings representan texto como vectores numericos que capturan significado",
    "RAG combina busqueda semantica con generacion de texto para respuestas fundamentadas",
]

# Metadata: area tematica de cada documento
areas = [
    "programacion", "programacion", "programacion",
    "datos", "datos", "datos",
    "visualizacion", "visualizacion", "visualizacion",
    "negocio", "negocio", "negocio",
    "ml", "ml", "ml",
]

# Insertar documentos con metadata
collection.add(
    documents=documents,
    metadatas=[{"area": area} for area in areas],
    ids=[f"doc_{i}" for i in range(len(documents))],
)

print(f"Coleccion creada con {collection.count()} documentos")

## Seccion 2 — Busquedas semanticas

Ahora viene lo interesante: vamos a buscar documentos **por significado**, no por palabras exactas.

Observa como ChromaDB encuentra documentos relevantes aunque no compartan palabras con la consulta.

In [ ]:
# 4 consultas semanticas — ninguna usa las palabras exactas de los documentos
queries = [
    "¿como programar para analizar informacion?",
    "herramientas para hacer graficas y reportes",
    "inteligencia artificial y aprendizaje automatico",
    "¿como mejorar las decisiones en una empresa?",
]

for query in queries:
    results = collection.query(query_texts=[query], n_results=3)

    print(f"\n🔍 Query: \"{query}\"")
    print("-" * 60)
    for i, (doc, dist, meta) in enumerate(
        zip(results["documents"][0], results["distances"][0], results["metadatas"][0])
    ):
        print(f"  {i+1}. [{meta['area']}] (dist: {dist:.4f}) {doc}")

### ¿Que observas?

- La query "programar para analizar informacion" encuentra documentos sobre Python y SQL sin usar esas palabras
- "Graficas y reportes" encuentra Tableau/Power BI y dashboards
- Las **distancias** mas bajas = mas similares semanticamente

**Esto es imposible con SQL `LIKE`** — necesitarias adivinar todas las palabras posibles.

## Seccion 3 — Embeddings visuales: PCA

Los embeddings son vectores de muchas dimensiones (384+). Para verlos, los reducimos a 2D con **PCA** (Principal Component Analysis).

Documentos sobre temas similares deberian quedar **cerca** en el grafico.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# Obtener los embeddings almacenados
all_data = collection.get(include=["embeddings", "metadatas", "documents"])
embeddings = np.array(all_data["embeddings"])
labels = [m["area"] for m in all_data["metadatas"]]

# Reducir a 2D con PCA
pca = PCA(n_components=2)
coords_2d = pca.fit_transform(embeddings)

# Colores del design system del workshop
color_map = {
    "programacion": "#2DD4BF",  # Teal
    "datos": "#FF6B6B",         # Coral
    "visualizacion": "#6C5CE7", # Purple
    "negocio": "#FFB347",       # Orange
    "ml": "#4ECDC4",            # Cyan
}

plt.figure(figsize=(10, 7))
plt.style.use("dark_background")

for area in color_map:
    mask = [l == area for l in labels]
    pts = coords_2d[mask]
    plt.scatter(pts[:, 0], pts[:, 1], c=color_map[area], label=area, s=120, edgecolors="white", linewidth=0.5)

# Anotar cada punto con texto recortado
for i, doc in enumerate(all_data["documents"]):
    short = doc[:35] + "..." if len(doc) > 35 else doc
    plt.annotate(short, (coords_2d[i, 0], coords_2d[i, 1]),
                 fontsize=6, color="white", alpha=0.7,
                 textcoords="offset points", xytext=(5, 5))

plt.title("Embeddings en 2D (PCA) — Documentos de BI", fontsize=14, fontweight="bold")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} varianza)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} varianza)")
plt.legend(title="Area", loc="best")
plt.tight_layout()
plt.show()

In [ ]:
# Alternativa: t-SNE (mejor para separar clusters, pero no preserva distancias globales)
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, random_state=42, perplexity=5)
coords_tsne = tsne.fit_transform(embeddings)

plt.figure(figsize=(10, 7))
plt.style.use("dark_background")

for area in color_map:
    mask = [l == area for l in labels]
    pts = coords_tsne[mask]
    plt.scatter(pts[:, 0], pts[:, 1], c=color_map[area], label=area, s=120, edgecolors="white", linewidth=0.5)

for i, doc in enumerate(all_data["documents"]):
    short = doc[:35] + "..." if len(doc) > 35 else doc
    plt.annotate(short, (coords_tsne[i, 0], coords_tsne[i, 1]),
                 fontsize=6, color="white", alpha=0.7,
                 textcoords="offset points", xytext=(5, 5))

plt.title("Embeddings en 2D (t-SNE) — Documentos de BI", fontsize=14, fontweight="bold")
plt.legend(title="Area", loc="best")
plt.tight_layout()
plt.show()

## Seccion 4 — Visualiza una consulta en el espacio de embeddings

Ahora vamos a ver **donde cae tu consulta** en relacion a los documentos, y **lineas** hacia los resultados mas cercanos.

In [ ]:
# Elegir una consulta
query_text = "herramientas para analizar datos de clientes"

# Buscar y obtener el embedding de la consulta
results = collection.query(
    query_texts=[query_text],
    n_results=3,
    include=["documents", "distances", "embeddings"],
)

# Obtener IDs de resultados para marcarlos
result_docs = results["documents"][0]
result_dists = results["distances"][0]

# Generar embedding de la query usando la misma funcion de ChromaDB
# ChromaDB usa all-MiniLM-L6-v2 por defecto
from chromadb.utils.embedding_functions import DefaultEmbeddingFunction

ef = DefaultEmbeddingFunction()
query_emb = np.array(ef([query_text]))

# Combinar con los embeddings existentes para PCA conjunto
all_embs = np.vstack([embeddings, query_emb])
pca2 = PCA(n_components=2)
all_coords = pca2.fit_transform(all_embs)

doc_coords = all_coords[:-1]
query_coord = all_coords[-1]

# Encontrar indices de los resultados
result_indices = []
for rdoc in result_docs:
    for j, d in enumerate(all_data["documents"]):
        if d == rdoc:
            result_indices.append(j)
            break

# Graficar
plt.figure(figsize=(10, 7))
plt.style.use("dark_background")

# Documentos normales (gris)
plt.scatter(doc_coords[:, 0], doc_coords[:, 1], c="#808080", s=60, alpha=0.4, label="Documentos")

# Resultados destacados
for idx in result_indices:
    plt.scatter(doc_coords[idx, 0], doc_coords[idx, 1], c="#2DD4BF", s=150, edgecolors="white", linewidth=1.5, zorder=5)
    plt.plot([query_coord[0], doc_coords[idx, 0]], [query_coord[1], doc_coords[idx, 1]],
             color="#2DD4BF", linewidth=1.5, alpha=0.6, linestyle="--")

# Query point
plt.scatter(query_coord[0], query_coord[1], c="#FF6B6B", s=200, marker="*", edgecolors="white", linewidth=1.5, zorder=10, label="Tu consulta")

# Anotar resultados
for i, idx in enumerate(result_indices):
    short = all_data["documents"][idx][:40] + "..."
    plt.annotate(f"#{i+1}: {short}", (doc_coords[idx, 0], doc_coords[idx, 1]),
                 fontsize=7, color="#2DD4BF", fontweight="bold",
                 textcoords="offset points", xytext=(8, 8))

plt.annotate(f'Query: "{query_text}"', (query_coord[0], query_coord[1]),
             fontsize=8, color="#FF6B6B", fontweight="bold",
             textcoords="offset points", xytext=(10, -15))

plt.title("Consulta en el espacio de embeddings", fontsize=14, fontweight="bold")
plt.legend(loc="best")
plt.tight_layout()
plt.show()

print(f"\nResultados para: \"{query_text}\"")
for i, (doc, dist) in enumerate(zip(result_docs, result_dists)):
    print(f"  {i+1}. (dist: {dist:.4f}) {doc}")

## Seccion 5 — RAG: Retrieval-Augmented Generation

RAG = **buscar contexto relevante** (ChromaDB) + **generar respuesta** (LLM).

Primero vamos a hacer RAG **sin LLM** (solo retrieval), y despues con together.ai si tienen API key.

In [ ]:
def ask_rag(question, n_results=3, api_key=None):
    """
    RAG: busca contexto en ChromaDB y genera respuesta.
    Si no hay API key, muestra solo el contexto recuperado.
    """
    # Paso 1: Retrieval — buscar documentos relevantes
    results = collection.query(query_texts=[question], n_results=n_results)
    context_docs = results["documents"][0]
    context_dists = results["distances"][0]

    print(f"📋 Pregunta: {question}")
    print(f"\n📚 Contexto recuperado (top {n_results}):")
    for i, (doc, dist) in enumerate(zip(context_docs, context_dists)):
        print(f"  {i+1}. (dist: {dist:.4f}) {doc}")

    # Paso 2: Generation — si hay API key, usar LLM
    if api_key:
        try:
            from together import Together

            client_llm = Together(api_key=api_key)
            context = "\n".join(f"- {doc}" for doc in context_docs)
            prompt = (
                f"Eres un asistente de Inteligencia de Negocios. "
                f"Responde la pregunta usando SOLO el contexto proporcionado. "
                f"Si no puedes responder con el contexto, dilo.\n\n"
                f"Contexto:\n{context}\n\n"
                f"Pregunta: {question}\n\n"
                f"Respuesta:"
            )

            response = client_llm.chat.completions.create(
                model="meta-llama/Llama-3.3-70B-Instruct-Turbo",
                messages=[{"role": "user", "content": prompt}],
                max_tokens=300,
            )
            answer = response.choices[0].message.content
            print(f"\n🤖 Respuesta del LLM:\n{answer}")
        except ImportError:
            print("\n⚠️ Instala together: !pip install together")
        except Exception as e:
            print(f"\n⚠️ Error con LLM: {e}")
    else:
        print("\n💡 Sin API key — mostrando solo retrieval.")
        print("   Para generar respuestas con LLM:")
        print('   ask_rag("tu pregunta", api_key="tu-api-key-de-together")')

    print("\n" + "=" * 60)

In [ ]:
# Probar RAG sin LLM (solo retrieval)
ask_rag("¿Que herramientas necesito para empezar en ciencia de datos?")
ask_rag("¿Como puedo presentar resultados a mi jefe?")
ask_rag("¿Que es lo mas nuevo en inteligencia artificial?")

In [ ]:
# OPCIONAL: RAG con LLM (necesitas API key de together.ai — es gratis)
# Descomenta y reemplaza con tu API key:

# !pip install -q together
# TOGETHER_API_KEY = "tu-api-key-aqui"  # Obtener en: https://api.together.xyz
# ask_rag("¿Que debo aprender para conseguir trabajo como Data Analyst?", api_key=TOGETHER_API_KEY)

## Ejercicios

Ahora te toca a ti. Intenta estos retos:

### 1. Agrega mas documentos
Añade 5 documentos nuevos sobre un tema que te interese (deportes, musica, cocina...) y busca con queries semanticas.

```python
collection.add(
    documents=["tu documento aqui"],
    metadatas=[{"area": "tu_area"}],
    ids=["doc_nuevo_1"],
)
```

### 2. Busqueda cross-language
Intenta buscar en ingles sobre la coleccion en español. ¿Funciona?

```python
results = collection.query(query_texts=["data visualization tools"], n_results=3)
```

### 3. Filtros por metadata
Busca solo en documentos de un area especifica:

```python
results = collection.query(
    query_texts=["programacion"],
    n_results=3,
    where={"area": "ml"},  # solo buscar en documentos de ML
)
```

### 4. Visualiza tus documentos nuevos
Vuelve a correr la celda de PCA despues de agregar documentos. ¿Donde aparecen?

## Conclusiones clave

| Concepto | Lo que aprendiste |
|----------|------------------|
| **Embeddings** | Los textos se convierten en vectores numericos que capturan significado |
| **Busqueda semantica** | Encuentra documentos por significado, no por palabras exactas |
| **Distancia** | Menor distancia = mas similitud semantica |
| **PCA / t-SNE** | Tecnicas para visualizar vectores de alta dimension en 2D |
| **RAG** | Busqueda semantica + LLM = respuestas fundamentadas en tus datos |

---

**Siguiente paso**: Explora los labs del repositorio para practicar con Elasticsearch y construir un mini-RAG completo.

📁 `github.com/HesusG/mas-alla-de-sql`